In [1]:
import os
import re
import ast
import numpy as np
import pandas as pd


# ── Configuration ──────────────────────────────────────────────────────────────
CSV_PATH = r"E:\Dissertation\Data\GreenbergZhao_river_datasheet.csv"
OUTPUT_PATH = r"E:\Dissertation\Data\RiverMapping\Mobility\GreenbergZhao_ureach_tstor_summarystats.csv"
# ──────────────────────────────────────────────────────────────────────────────


def compute_distribution_stats(values, prefix):
    """
    Computes descriptive statistics for a 1D array of values.

    Args:
        values (np.ndarray): Array of floats, may contain inf or NaN.
        prefix (str): Column name prefix (e.g. 'tstor' or 'ureach').

    Returns:
        dict: Statistics keyed by prefixed column names.
    """
    return {
        f"{prefix}_min":    np.min(values),
        f"{prefix}_1qt":    np.percentile(values, 25),
        f"{prefix}_median": np.median(values),
        f"{prefix}_3qt":    np.percentile(values, 75),
        f"{prefix}_max":    np.max(values),
        f"{prefix}_mean":   np.mean(values),
        f"{prefix}_std":    np.std(values),
    }


def compile_reach_statistics(csv_path, output_path):
    """
    Compiles per-reach T_stor and u_reach distribution statistics for all rivers
    and reaches listed in a master CSV file.

    Args:
        csv_path (str):   Path to the master CSV file.
        output_path (str): Path to write the output statistics CSV.
    """
    config_data = pd.read_csv(csv_path)
    all_rows = []

    for _, row in config_data.iterrows():
        river_name        = row['river_name']
        working_directory = row['working_directory']

        tstor_dir  = os.path.join(working_directory, "RiverMapping", "Mobility", river_name, "Tstor_distributions")
        treach_dir = os.path.join(working_directory, "RiverMapping", "Mobility", river_name, "treach_distributions")
        ureach_dir = os.path.join(working_directory, "RiverMapping", "Mobility", river_name, "ureach_distributions")

        # Identify available reaches from Tstor distributions
        if not os.path.isdir(tstor_dir):
            print(f"Tstor directory not found, skipping {river_name}: {tstor_dir}")
            continue

        tstor_files = [f for f in os.listdir(tstor_dir) if f.endswith(".csv") and f.startswith("Reach_")]
        if not tstor_files:
            print(f"No Tstor files found, skipping {river_name}: {tstor_dir}")
            continue

        reach_numbers = sorted([int(f.split("_")[1]) for f in tstor_files])

        for reach_number in reach_numbers:
            reach_row = {"river_name": river_name, "ds_order": reach_number}

            # ── T_stor statistics ──────────────────────────────────────────
            tstor_path = os.path.join(tstor_dir, f"Reach_{reach_number}_Tstor_distribution.csv")
            if not os.path.isfile(tstor_path):
                print(f"Tstor file not found, skipping Reach {reach_number} of {river_name}")
                continue
            tstor_vals = pd.read_csv(tstor_path).iloc[:, 0].dropna().values
            reach_row.update(compute_distribution_stats(tstor_vals, "tstor"))

            # ── u_reach statistics ─────────────────────────────────────────
            ureach_path = os.path.join(ureach_dir, f"Reach_{reach_number}_ureach_distribution.csv")
            if not os.path.isfile(ureach_path):
                print(f"ureach file not found, skipping Reach {reach_number} of {river_name}")
                continue
            ureach_vals = pd.read_csv(ureach_path).iloc[:, 0].values

            # Flag reaches with any inf or NaN in the ureach distribution
            has_unusual = bool(np.any(~np.isfinite(ureach_vals)))
            reach_row["ureach_unusual_values"] = has_unusual

            reach_row.update(compute_distribution_stats(ureach_vals, "ureach"))

            all_rows.append(reach_row)

    # Write output
    output_df = pd.DataFrame(all_rows)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    output_df.to_csv(output_path, index=False)
    print(f"\nStatistics saved to: {output_path}")
    print(f"Total reaches processed: {len(output_df)}")


# ── Run ────────────────────────────────────────────────────────────────────────
compile_reach_statistics(CSV_PATH, OUTPUT_PATH)

C:\Users\huckr\miniconda3\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\core\_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\core\_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\core\_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\core\_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\


Statistics saved to: E:\Dissertation\Data\RiverMapping\Mobility\GreenbergZhao_ureach_tstor_summarystats.csv
Total reaches processed: 121


C:\Users\huckr\miniconda3\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\core\_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\core\_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\lib\function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
C:\Users\huckr\miniconda3\Lib\site-packages\numpy\core\_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
